# 02 — GREmLN gene embeddings from the masked GENIE3 prior

Runs the **GREmLN** foundation model (frozen checkpoint, zero-shot) to produce per-gene embeddings,
using the masked CD4 GENIE3 graph (notebook 01) as its **regulatory prior**. Key design point: the
expression fed to GREmLN is the **pre-ComBat log1p-CPM** of the same 50k cells, because GREmLN's
tokenizer needs the rank/zero structure of near-count data (ComBat output has negatives / few true
zeros). GPU inference is behind `REGENERATE`; by default we **reuse** the saved embeddings and audit
coverage. Obtain the checkpoint per `data/README.md`.

In [ ]:
import sys, json, platform
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import bench_utils as bu
import yaml
CFG = yaml.safe_load((bu.repo_root() / "config" / "benchmark_config.yaml").read_text())
REGENERATE = False       # True runs GREmLN inference (GPU). False reuses saved embeddings.
REG = bu.repo_root() / "results" / "model_registry"; REG.mkdir(parents=True, exist_ok=True)
print("GREmLN cfg:", {k: CFG["gremln"][k] for k in ("embedding_dim", "expression_input", "prior_graph")})

## 1. Provenance: checkpoint, submodule commit, expression input, graph prior

In [ ]:
def pkg(n):
    try:
        from importlib.metadata import version; return version(n)
    except Exception: return "unknown"
def git_commit(path):
    import subprocess
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(path),
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception: return "unknown"
gremln_commit = git_commit(bu.repo_root() / "third_party" / "GREmLN")
print("GREmLN submodule commit:", gremln_commit)
print("checkpoint md5 (config):", CFG["gremln"]["checkpoint_md5"])

## 2. GREmLN inference (GPU, guarded): vocab → RegulatoryNetwork(prior) → tokenizer → embeddings

The real inference call is inlined below. It loads the frozen `GDTransformer`, builds the
`GraphTokenizer` from the GREmLN gene vocabulary and the masked GENIE3 graph as `RegulatoryNetwork`,
and runs a single gene-embedding forward pass (`get_gene_embeddings`), mapping Ensembl→HUGO.

In [ ]:
if REGENERATE:
    import anndata as ad, torch
    sys.path.insert(0, str(bu.repo_root() / "third_party" / "GREmLN"))
    from scGraphLLM import GeneVocab, GraphTokenizer, InferenceDataset, RegulatoryNetwork
    from scGraphLLM.config import graph_kernel_attn_3L_4096
    from scGraphLLM.inference import get_gene_embeddings
    from scGraphLLM.models import GDTransformer
    adata = ad.read_h5ad(bu.artifact("gremln_inputs") / "expr.h5ad")     # pre-ComBat log1p-CPM, Ensembl var
    vocab = GeneVocab.from_csv(bu.repo_root() / "third_party/GREmLN/scGraphLLM/resources/gene_vocab.csv",
                               gene_col="gene_name", node_col="idx")
    network = RegulatoryNetwork.from_csv(bu.artifact("gremln_inputs") / "graph.tsv", sep="\t",
                                         reg_name="regulator.values", tar_name="target.values",
                                         wt_name="mi.values", lik_name="log.p.values")
    model = GDTransformer.load_from_checkpoint(str(bu.artifact("gremln_checkpoint")),
                                               config=graph_kernel_attn_3L_4096).eval().cuda()
    ds = InferenceDataset(expression=adata.to_df(), tokenizer=GraphTokenizer(vocab=vocab, network=network))
    x_gene = get_gene_embeddings(ds, model, vocab, batch_size=256)       # per-gene mean embeddings
    emb = ad.AnnData(x_gene.values, obs=adata.var.loc[x_gene.index].copy())
    emb.obs["hugo"] = emb.obs.get("hugo_symbol", pd.Series(emb.obs_names)).astype(str)
    emb.write_h5ad(bu.artifact("gremln_emb"))
else:
    print("REGENERATE=False -> reuse saved GREmLN embeddings (inference code above for provenance).")

## 3. Load embeddings + coverage audit + sanitized model registry

In [ ]:
genes, X = bu.load_gremln_embeddings()
tfs = bu.load_tfs(); edges = bu.load_edges(); seeds, seed_dir, _ = bu.load_seeds()
gremln_tfs = set(genes) & tfs; genie3_tfs = set(edges["regulator"]) & tfs
common = gremln_tfs & genie3_tfs
audit = pd.DataFrame([
    ("genes_embedded_by_gremln", len(genes)), ("embedding_dim", X.shape[1]),
    ("tfs_embedded_by_gremln", len(gremln_tfs)), ("tfs_in_genie3_graph", len(genie3_tfs)),
    ("common_tf_universe", len(common)), ("btla_seed_panel_size", len(seeds)),
    ("seeds_available_to_gremln", len(set(seeds) & set(genes))),
    ("seed_tfs_in_common_universe", len(set(seeds) & common)),
], columns=["metric", "value"])
print(audit.to_string(index=False))
registry = pd.DataFrame([
    ("model_name", "GREmLN_CD4_masked_raw_prior"),
    ("gremln_checkpoint", "model.ckpt"), ("gremln_checkpoint_md5", CFG["gremln"]["checkpoint_md5"]),
    ("gremln_submodule_commit", gremln_commit),
    ("python_version", platform.python_version()),
    ("numpy_version", pkg("numpy")), ("pandas_version", pkg("pandas")),
    ("anndata_version", pkg("anndata")), ("torch_version", pkg("torch")),
    ("expression_input", CFG["gremln"]["expression_input"]),
    ("prior_graph", CFG["gremln"]["prior_graph"]),
    ("prior_graph_md5", bu.md5(bu.artifact("genie3_edges"))),
    ("embedding_h5ad_md5", bu.md5(bu.artifact("gremln_emb"))),
    ("n_genes_embedded", len(genes)), ("embedding_dim", X.shape[1]),
    ("csls_params", CFG["scoring"]["gremln"]), ("random_seed", CFG["seed"]),
    ("device", "cuda"), ("inference_params", CFG["gremln"]["inference"]),
], columns=["field", "value"])
registry.to_csv(REG / "gremln_model_registry.csv", index=False)
audit.to_csv(REG / "gremln_gene_universe_audit.csv", index=False)
registry